In [12]:
import csv
import pandas as pd
import re

In [13]:
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

In [14]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"

last_file="anonymous_answer_2026-07-24.csv"

In [17]:
df0 = pd.read_csv('buparis8_chercheurs_besoins_accompagnement_7-24-2026_9_54.csv', sep =";")


In [18]:
df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
#df_col["name"] = df_col.name.apply(lambda row : row.replace(':', ''))
df_col = df_col.iloc[:143]
df_col

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
138,151. Langue,q50_langue,17_metadonnee,False,simple_nominal,False,object,NaN,NaN,q50,NaN
139,152. Progression,q51_progression,17_metadonnee,False,NaN,False,object,NaN,NaN,q51,NaN
140,153. Dernière question saisie,q52_dernière_question_saisie,17_metadonnee,False,simple_nominal,False,float64,NaN,NaN,q52,NaN
141,154. Origine,q53_origine,17_metadonnee,False,simple_nominal,False,object,NaN,NaN,q53,NaN


In [19]:
dict_new_col = dict(zip(df_col.name, df_col.label.str.lower()))

df = df0.rename(columns=dict_new_col)
df.columns

Index(['qno_obs', 'q1_so_principles', 'q2_hal_depot', 'q3_octavi_depot',
       'q4_diff_data', 'q4_autres_entrepots', 'q5_r', 'q5_python', 'q5_excel',
       'q5_julia',
       ...
       'q45_clé', 'q46_date_de_saisie', 'q47_date_de_dernière_modification',
       'q48_date_de_dernier_enregistrement', 'q49_temps_de_saisie',
       'q50_langue', 'q51_progression', 'q52_dernière_question_saisie',
       'q53_origine', 'q54_appareil_utilisé_pour_la_saisie'],
      dtype='object', length=156)

In [20]:
df_col.loc[df_col.label== "q45_clé"]

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
133,146. Clé,q45_clé,12_descript_repondant,False,string,False,object,49.0,4/ Si vous souhaitez un suivi ou un échange à ...,q45,NaN


In [21]:
list_personal_data = [ x.lower() for x in df_col.label.loc[df_col.personal_data==True]]
list_personal_data

['q42_nom_', 'q42_prénom_', 'q43_adresse_e-mail_']

In [22]:
df1 = df.drop(columns= list_personal_data)
df1

,qno_obs,q1_so_principles,q2_hal_depot,q3_octavi_depot,q4_diff_data,q4_autres_entrepots,q5_r,q5_python,q5_excel,q5_julia,...,q45_clé,q46_date_de_saisie,q47_date_de_dernière_modification,q48_date_de_dernier_enregistrement,q49_temps_de_saisie,q50_langue,q51_progression,q52_dernière_question_saisie,q53_origine,q54_appareil_utilisé_pour_la_saisie
0,1,"Oui, un peu","Oui, une fois",Non,Non,NaN,Non,Non,Non,Non,...,PS8W-TJ9P,09/03/2026 11:01:53,NaN,09/03/2026 11:12:04,"611,0190107",FR,Terminé,NaN,Direct,PC
1,2,"Oui, un peu","Oui, une fois",Non,Non,NaN,Non,Non,Non,Non,...,GNEG-5QXQ,10/03/2026 11:11:19,NaN,10/03/2026 11:17:47,"387,3420906",FR,Terminé,NaN,Direct,PC
2,3,"Oui, tout à fait","Oui, plusieurs fois",Je ne connais pas Octaviana,Non,NaN,Non,Non,Oui,Non,...,FCCB-GMYA,10/03/2026 11:10:00,NaN,10/03/2026 11:19:57,"596,9405636",FR,Terminé,NaN,Direct,PC
3,4,"Oui, un peu","Oui, une fois",Non,"Autre, précisez",OSF,Oui,Non,Non,Non,...,CV3G-95CT,10/03/2026 11:14:07,NaN,10/03/2026 11:23:25,"558,0371672",FR,Terminé,NaN,Direct,PC
4,5,"Oui, un peu","Oui, systématiquement",Non,Non,NaN,Non,Non,Oui,Non,...,9EAP-NB4B,10/03/2026 11:18:54,NaN,10/03/2026 11:28:09,"554,5038942",FR,Terminé,NaN,Direct,PC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,117,"Oui, un peu","Oui, plusieurs fois",Non,"Autre, précisez",OSF,Oui,Oui,Oui,Non,...,MLSN-NBUS,28/04/2026 17:33:55,NaN,28/04/2026 17:49:27,"932,1846343",FR,Terminé,NaN,Direct,PC
117,118,"Oui, un peu","Oui, systématiquement",Non,Non,NaN,Oui,Non,Oui,Non,...,WAR9-LFJP,29/04/2026 14:59:02,NaN,29/04/2026 15:10:56,"714,0441946",FR,Terminé,NaN,Direct,PC
118,119,"Oui, un peu","Oui, plusieurs fois",Non,Non,NaN,Oui,Non,Oui,Non,...,HRZG-6JGE,05/05/2026 09:36:22,NaN,05/05/2026 09:52:15,"953,6974524",FR,Terminé,NaN,Direct,PC
119,120,Non,"Oui, plusieurs fois",Non;Je ne connais pas Octaviana,"Autre, précisez",Mendeley data,Non,Oui,Oui,Non,...,32TW-JLY5,28/05/2026 10:01:22,NaN,28/05/2026 10:21:58,"1235,8811168",FR,Terminé,NaN,Direct,PC


In [23]:
df1.loc[df1.q45_clé.isin(["H5X6-KL2Z",'PS8W-TJ9P'])].drop_duplicates(subset="q45_clé")#.merge(list_affil, on = ["q45_clé", "q44_ufr_labo"], how = "left")

,qno_obs,q1_so_principles,q2_hal_depot,q3_octavi_depot,q4_diff_data,q4_autres_entrepots,q5_r,q5_python,q5_excel,q5_julia,...,q45_clé,q46_date_de_saisie,q47_date_de_dernière_modification,q48_date_de_dernier_enregistrement,q49_temps_de_saisie,q50_langue,q51_progression,q52_dernière_question_saisie,q53_origine,q54_appareil_utilisé_pour_la_saisie
0,1,"Oui, un peu","Oui, une fois",Non,Non,NaN,Non,Non,Non,Non,...,PS8W-TJ9P,09/03/2026 11:01:53,NaN,09/03/2026 11:12:04,"611,0190107",FR,Terminé,NaN,Direct,PC
57,58,"Oui, un peu",Non,Non,Non,NaN,Non,Non,Oui,Non,...,H5X6-KL2Z,23/03/2026 14:09:16,NaN,23/03/2026 15:10:28,"3672,1628372",FR,Terminé,NaN,Direct,PC


In [24]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/{last_file}", "w") as file_out:
    df1.to_csv(file_out, sep=",", index= False)